In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')
print(f'TensorLy tenalg backend: {tl.tenalg.get_backend()}')

env: CUPY_ACCELERATORS=cutensor,cub
TensorLy backend: cupy
TensorLy tenalg backend: einsum


In [2]:
from moabb.paradigms import FilterBankMotorImagery
from moabb.datasets import *
from hoda.tensorize import fh_envelope


paradigm = FilterBankMotorImagery(resample=250)
dataset = AlexMI()
X, y, meta = paradigm.get_data(
     dataset=dataset, 
     subjects=[1],
)
X = fh_envelope(X, sfreq=250, target_sfreq=32)
X = tl.tensor(X)
X.shape

Choosing from all possible events
/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~7.5 MiB, data loaded,
 'right_hand': 20
 'feet': 20>

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~7.5 MiB, data loaded,
 'right_hand': 20
 'feet': 20>

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~7.5 MiB, data loaded,
 'right_hand': 20
 'feet': 20>

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~

(40, 6, 16, 96)

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import pandas as pd

x = X.flatten()
x = tl.to_numpy(x)
hist, bin_edges = np.histogram(x, bins=100000)
bins = (bin_edges[:-1] + bin_edges[1:]) / 2  # Compute the mean of each subsequent pair
df = pd.DataFrame({'bins':bins, 'count':hist})
px.histogram(df, x="bins", y="count")

In [64]:
from hoda.hoda import HODA

hoda = HODA(
        rank=1,
        max_iter=1024,
        tol=1e-6,
        shrinkage='lw',
        toeplitz=(2,),
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=True,
        forward=True,
        theta=None,
        refit_shrinkage=True,
)


In [65]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from hoda.util import solve_gevdh

plt.style.use('default')
%load_ext line_profiler

hoda.fit_backward(X,y)
df = pd.DataFrame(hoda.train_info_['backward'])
display(df)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


Backward HODA model rank=(1, 1, 1): 100%|██████████| 1024/1024 [00:19<00:00, 51.81it/s]
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:361: UserWarning:

Maximum number of iterations reached without convergence



,iteration,mode,flip,update,shrinkage,objective
0,1,1,1,0.981777,0.299621,28160.356818
1,1,2,2,0.772287,0.142676,38319.783010
2,1,3,3,0.986529,0.357062,9218.974165
3,2,1,4,0.489086,0.495982,28941.797558
4,2,2,5,0.429969,0.147059,20860.539440
...,...,...,...,...,...,...
3067,1023,2,3068,0.051107,0.141846,7099.778818
3068,1023,3,3069,0.048962,0.409060,2965.505975
3069,1024,1,3070,0.019957,0.743651,28051.099170
3070,1024,2,3071,0.059403,0.145532,7345.483779


In [66]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

,iteration,mode,flip,update,shrinkage,objective
0,1,1,1,0.981777,0.299621,28160.356818
1,1,2,2,0.772287,0.142676,38319.783010
2,1,3,3,0.986529,0.357062,9218.974165
3,2,1,3,0.489086,0.495982,28941.797558
4,2,2,4,0.429969,0.147059,20860.539440
...,...,...,...,...,...,...
3067,1023,2,2046,0.051107,0.141846,7099.778818
3068,1023,3,2047,0.048962,0.409060,2965.505975
3069,1024,1,2047,0.019957,0.743651,28051.099170
3070,1024,2,2048,0.059403,0.145532,7345.483779


In [67]:
import plotly.express as px

if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_tr', log_y=False)
    fig.show()

In [68]:
px.line(df, x='iteration', y='update', log_y=True, color='mode')


In [69]:
px.line(df, x='flip', y='shrinkage', log_y=False, color='mode')

In [70]:
px.line(df, x='flip', y='objective', log_y=False, color='mode')

In [71]:
hoda.fit_forward(X,y)
df = pd.DataFrame(hoda.train_info_['forward'])
df

Forward model :   1%|          | 7/1023 [00:00<00:03, 291.25it/s]


,iteration,mode,flip,update,lambda_
0,1,1,1,5.060220e-01,0.0
1,1,2,2,2.885951e+00,0.0
2,1,3,3,3.033178e+00,0.0
3,2,1,4,1.050279e+00,0.0
4,2,2,5,3.477856e-01,0.0
5,2,3,6,1.165242e-01,0.0
6,3,1,7,9.606788e-02,0.0
7,3,2,8,2.577910e-02,0.0
8,3,3,9,1.018755e-02,0.0
9,4,1,10,7.519143e-03,0.0


In [72]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

,iteration,mode,flip,update,lambda_
0,1,1,1,5.060220e-01,0.0
1,1,2,2,2.885951e+00,0.0
2,1,3,3,3.033178e+00,0.0
3,2,1,3,1.050279e+00,0.0
4,2,2,4,3.477856e-01,0.0
5,2,3,5,1.165242e-01,0.0
6,3,1,5,9.606788e-02,0.0
7,3,2,6,2.577910e-02,0.0
8,3,3,7,1.018755e-02,0.0
9,4,1,7,7.519143e-03,0.0


In [73]:
if hoda.extra_train_info:
    x.line(df, x='flip', y='mse', log_y=True)

In [74]:
px.line(df, x='flip', y='update', log_y=True, color='mode')

In [75]:
from sklearn.feature_selection import SelectFpr, SelectFwe, SelectFdr
import numpy as np

Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))

select = SelectFdr(alpha=.05)
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'p_value': select.pvalues_,
    'significant': select.get_support()
})
df

,feature,F,p_value,significant
0,0,251.721206,2.378811e-18,True


In [76]:
fig = px.bar(df, x='feature', y='F', color='significant', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [77]:
from sklearn.manifold import TSNE
x_viz = TSNE(n_components=2).fit_transform(xt)
px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)

ValueError: n_components=2 must be between 1 and min(n_samples, n_features)=1 with svd_solver='randomized'